In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score

# Load and clean data
df = pd.read_csv('Algerian_forest_fires_dataset (1).csv')

# Data cleaning
df = df.dropna().reset_index(drop=True)
df = df.drop(124).reset_index(drop=True)  # Remove header row
df.columns = df.columns.str.strip()

# More robust data cleaning
def clean_numeric(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

# Convert data types with error handling
numeric_cols = ['Temperature', 'RH', 'Ws']
float_cols = ['Rain', 'FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']

for col in numeric_cols + float_cols:
    df[col] = df[col].apply(clean_numeric)
    
# Drop any remaining NA values
df = df.dropna()

# Now safely convert types
df[numeric_cols] = df[numeric_cols].astype(int)
df[float_cols] = df[float_cols].astype(float)

# Add region column
df.loc[:122, 'Region'] = 0  # Bejaia
df.loc[123:, 'Region'] = 1   # Sidi Bel-abbes
df['Region'] = df['Region'].astype(int)

# Clean target variable
df['Classes'] = (df['Classes'].str.strip()
                .replace({'not fire': 0, 'fire': 1})
                .astype(int))

# Feature Engineering
X = df.drop(['day', 'year', 'FWI', 'Classes'], axis=1)
y = df['FWI']

# Remove highly correlated features
def remove_correlated_features(dataset, threshold=0.85):
    corr_matrix = dataset.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    return dataset.drop(to_drop, axis=1)

X = remove_correlated_features(X)

# Train-test split and scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model training and evaluation
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'MAE': mean_absolute_error(y_test, y_pred),
        'R2': r2_score(y_test, y_pred)
    }

# Linear Regression
lr_results = evaluate_model(LinearRegression(), X_train_scaled, X_test_scaled, y_train, y_test)
print("Linear Regression Results:", lr_results)

# Ridge Regression
ridge_results = evaluate_model(Ridge(), X_train_scaled, X_test_scaled, y_train, y_test)
print("Ridge Regression Results:", ridge_results)

# Save cleaned data
df.to_csv('algerian_forest_fires_cleaned.csv', index=False)

C:\Users\MCS\AppData\Local\Temp\ipykernel_10436\3343460344.py:44: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({'not fire': 0, 'fire': 1})


Linear Regression Results: {'MAE': 0.5992992474856991, 'R2': 0.9828847931040533}
Ridge Regression Results: {'MAE': 0.6225340284370092, 'R2': 0.982063389022737}
